In [1]:
from pathlib import Path

import pandas as pd


# 중복 데이터 제거

In [ ]:
CSV_DIR = Path('../data/csv')
OUT_PATH = Path('../data/csv/merged_dedup.csv')

ID_COL = 'id'
ENCODING = 'utf-8-sig'

csv_files = sorted(CSV_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in: {CSV_DIR.resolve()}')

dfs: list[pd.DataFrame] = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path, encoding=ENCODING)
    df['source_file'] = csv_path.name
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

if ID_COL not in merged.columns:
    raise ValueError(f"'{ID_COL}' column not found.")

merged[ID_COL] = merged[ID_COL].astype(str).str.strip()


# 문자열 컬럼 기준으로 비어있지 않은 값 개수 계산
def filled_count(row: pd.Series) -> int:
    count = 0
    for v in row:
        if pd.notna(v) and str(v).strip() != '':
            count += 1
    return count


merged['filled_count'] = merged.apply(filled_count, axis=1)

# 정보 많은 row 우선
merged = merged.sort_values(
    by=['filled_count'],
    ascending=False,
    kind='stable',
)

# duplicate_mask = merged['id'].duplicated(keep=False)

## 2. 마스크를 적용해서 중복된 데이터만 필터링해서 보기
# merged[duplicate_mask].sort_values(by='id').head(2)

dedup = merged.drop_duplicates(subset=[ID_COL], keep='first').reset_index(drop=True)

dedup.to_csv(OUT_PATH, index=False, encoding=ENCODING)

print('before:', len(merged))
print('after :', len(dedup))
print('saved :', OUT_PATH.resolve())

before: 306422
after : 246179
saved : C:\PJ02\pj02_Daangn-marke\data\csv\merged_dedup.csv


In [ ]:
df = pd.read_csv('../data/csv/merged_dedup.csv')
df.head(1)

C:\Users\user\AppData\Local\Temp\ipykernel_339432\1774543463.py:1: DtypeWarning: Columns (0: error) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/csv/merged_dedup.csv")


,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,...,viewCount,sellerTemperature,imageCount,imageUrls,imageFile,imagePath,status_detail,error,source_file,filled_count
0,1xyhcgcovmk1,https://www.daangn.com/kr/buy-sell/%ED%94%8C%E...,10000.0,플라워 패턴 셔링 나시 미니 원피스,Ongoing,빨간색 바탕에 플라워 패턴이 들어간 나시 미니 원피스입니다. \n원피스 하단에 셔링...,2026-01-27 09:36:27.939000+09:00,2026-01-27 09:36:27.936000+09:00,4001781.0,낫띵헤픈스,...,16.0,41.6,3.0,['https://img.kr.gcp-karroter.net/origin/artic...,1xyhcgcovmk1.webp,/content/drive/MyDrive/data/images/민소매/1xyhcgc...,ok,ChunkedEncodingError(ProtocolError('Connection...,daangn_민소매.csv,26


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 246179 entries, 0 to 246178
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        246179 non-null  str    
 1   href                      246179 non-null  str    
 2   price                     245757 non-null  float64
 3   title                     246179 non-null  str    
 4   status                    246179 non-null  str    
 5   content                   246179 non-null  str    
 6   createdAt                 246179 non-null  str    
 7   boostedAt                 246179 non-null  str    
 8   user_dbId                 246177 non-null  float64
 9   user_nickname             246177 non-null  str    
 10  region_name_from_article  246179 non-null  str    
 11  region_id                 246179 non-null  int64  
 12  region_name               246179 non-null  str    
 13  region_in                 246179 non-null  str    
 14 

In [ ]:
df.to_parquet(r'../data/parquet/merged_dedup.parquet', engine='auto', index=False)